In [21]:
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config

bucket = "stocktwits-nyu"
prefix = "dataset/v1/data/csv/feature_wo_messages/"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != prefix:
            print(os.path.basename(key))

feature_wo_messages_000.csv
feature_wo_messages_001.csv
feature_wo_messages_002.csv
feature_wo_messages_003.csv
feature_wo_messages_004.csv
feature_wo_messages_005.csv
feature_wo_messages_006.csv
feature_wo_messages_007.csv
feature_wo_messages_008.csv
feature_wo_messages_009.csv
feature_wo_messages_010.csv
feature_wo_messages_011.csv
feature_wo_messages_012.csv
feature_wo_messages_013.csv
feature_wo_messages_014.csv
feature_wo_messages_015.csv
feature_wo_messages_016.csv
feature_wo_messages_017.csv
feature_wo_messages_018.csv
feature_wo_messages_019.csv
feature_wo_messages_020.csv
feature_wo_messages_021.csv
feature_wo_messages_022.csv
feature_wo_messages_023.csv
feature_wo_messages_024.csv
feature_wo_messages_025.csv
feature_wo_messages_026.csv
feature_wo_messages_027.csv
feature_wo_messages_028.csv
feature_wo_messages_029.csv
feature_wo_messages_030.csv
feature_wo_messages_031.csv
feature_wo_messages_032.csv
feature_wo_messages_033.csv
feature_wo_messages_034.csv
feature_wo_messages_

In [22]:
BASE_URL = "s3://stocktwits-nyu" # or local path BASE_URL="local_path"
CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"

In [23]:
import pandas as pd

dtype_map = {"sentiment": "object", "message_id": "object"}
base_url = f"{CSV_URL}/feature_wo_messages"

dfs = {}

for i in range(227, 248):
    url = f"{base_url}/feature_wo_messages_{i:03d}.csv"
    print("attempting to load", url)
    try:
        df = pd.read_csv(url, dtype=dtype_map, usecols=["user_id", "created_at", "sentiment", "symbol_list"])
        name = f"feature_wo_messages_{i:03d}.csv"
        dfs[name] = df
        print("loaded", name)
    except Exception as e:
        print("missing or unreadable:", url, "|", type(e).__name__)
        continue

attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_227.csv
loaded feature_wo_messages_227.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_228.csv
loaded feature_wo_messages_228.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_229.csv
loaded feature_wo_messages_229.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_230.csv
loaded feature_wo_messages_230.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_231.csv
loaded feature_wo_messages_231.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_232.csv
loaded feature_wo_messages_232.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_233.csv
loaded feature_wo_messages_233.csv
attemp

In [24]:
print("loaded files:", len(dfs))

combined_df = pd.concat(
    [df.assign(source_file=name) for name, df in dfs.items()],
    ignore_index=True
)

loaded files: 21


In [25]:
combined_df = combined_df.dropna(subset=["sentiment"])
combined_df = combined_df[combined_df["symbol_list"] != "[]"]

In [26]:
combined_df["created_at"] = pd.to_datetime(combined_df["created_at"])

In [27]:
import ast
combined_df = combined_df.drop(columns=["source_file"])
combined_df["symbol_list"] = combined_df["symbol_list"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

combined_df = combined_df.explode("symbol_list", ignore_index=True)

combined_df = combined_df.rename(columns={"symbol_list": "symbol"})

In [28]:
print(combined_df.head())

   user_id                created_at sentiment symbol
0   493490 2016-07-06 03:05:29+00:00   Bullish    SPY
1   626206 2016-07-06 03:05:42+00:00   Bearish   MEET
2   737354 2016-07-06 03:05:59+00:00   Bullish   BBVA
3   567474 2016-07-06 03:06:53+00:00   Bearish   TWLO
4   761351 2016-07-06 03:07:25+00:00   Bullish   VOYA


In [30]:
import requests

In [31]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://en.wikipedia.org/"
}

In [32]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
r = requests.get(url, headers=headers)
spx = pd.read_html(r.text)[0]

/tmp/ipykernel_12712/3083819388.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  spx = pd.read_html(r.text)[0]


In [33]:
spx_tickers = spx["Symbol"].tolist()

In [42]:
df_snp = combined_df[combined_df["symbol"].apply(lambda symbols: any(sym in spx_tickers for sym in symbols))]

In [43]:
df_snp = combined_df[combined_df["sentiment"].notna()]

In [44]:
df_snp["created_at"] = pd.to_datetime(df_snp["created_at"])

In [47]:
df_snp["sentiment"].value_counts()

sentiment
Bullish    6824642
Bearish    1479713
Name: count, dtype: int64

In [48]:
df.shape

(1061332, 4)

In [49]:
df_snp

,user_id,created_at,sentiment,symbol
0,493490,2016-07-06 03:05:29+00:00,Bullish,SPY
1,626206,2016-07-06 03:05:42+00:00,Bearish,MEET
2,737354,2016-07-06 03:05:59+00:00,Bullish,BBVA
3,567474,2016-07-06 03:06:53+00:00,Bearish,TWLO
4,761351,2016-07-06 03:07:25+00:00,Bullish,VOYA
...,...,...,...,...
8304350,1233706,2017-11-02 22:08:53+00:00,Bullish,ASXC
8304351,332727,2017-11-02 22:08:58+00:00,Bullish,ALNY
8304352,185862,2017-11-02 22:08:59+00:00,Bullish,BHC
8304353,465024,2017-11-02 22:09:00+00:00,Bullish,AAPL


In [50]:
df_snp.to_csv("cleaned_data.csv", index=False)